# Overview
I think the `load_config` function in astera is a good sample function for Dylan...there are a few variables,
and some with nested structures (which is the key test case to make sure we have "dependencies" defined first)

In [1]:
!ls ~/exp_builds/astera.exp/rundata/run1/0.fighter
# !ls ~/exp_builds/astera.exp

0.fighter	   function_params.csv	      locals.stats.csv
0.fighter.debug    function_params.stats.csv  _raw_debug_locals.csv
ast_dumps	   functions.csv	      _raw_dwarf_locals.csv
fighter		   ghidra_ast.debug.json      _raw_stripped_locals.csv
fighter.debug	   ghidra_ast.json
fighter.dwarf.sdb  locals.csv


In [2]:
from pathlib import Path
import pandas as pd

bin_folder = Path.home()/'exp_builds/astera.exp/rundata/run1/0.fighter'

funcs_df = pd.read_csv(bin_folder/'functions.csv')
locals_df = pd.read_csv(bin_folder/'locals.csv')

raw_debug = pd.read_csv(bin_folder/'_raw_debug_locals.csv')
raw_dwarf = pd.read_csv(bin_folder/'_raw_dwarf_locals.csv')
raw_strip = pd.read_csv(bin_folder/'_raw_stripped_locals.csv')

In [3]:
addr = funcs_df[funcs_df.FunctionName_Debug=='init_game'].FunctionStart.iloc[0]
# pd.set_option('display.expand_frame_repr', False)
# pd.options.display.max_columns = 10
with pd.option_context('display.max_rows', None, 'display.max_columns', None,'display.max_colwidth', 20):
    print(len(locals_df[locals_df.FunctionStart==addr]))

46


In [4]:
raw_debug[raw_debug.FunctionStart==addr]

,FunctionStart,Name,Signature,Type,LocType,LocRegName,LocOffset,TypeCategory,TypeSeq
411,1351728,lVar1,"15,1739",int64,unique,NaN,49792,BUILTIN,int64
412,1351728,afVar2,"144,144,494",float[2],unique,NaN,268435571,ARR,"ARR,float"
413,1351728,lVar3,"400,408,408,408",int64,register,rcx,8,BUILTIN,int64
414,1351728,ppVar4,"405,408,408,408",player_t*,register,rdi,56,PTR,"PTR,STRUCT"
415,1351728,in_FS_OFFSET,"15,1739",int64,register,fs_offset,272,BUILTIN,int64
416,1351728,bVar5,"4,408",uchar,register,df,522,BUILTIN,uchar
417,1351728,extraout_XMM0_Qa,494,float[2],register,xmm0_qa,4608,ARR,"ARR,float"
418,1351728,extraout_XMM0_Qa_00,494,float[2],register,xmm0_qa,4608,ARR,"ARR,float"
419,1351728,in_XMM1_Da,144,uint32,register,xmm1_da,4640,BUILTIN,uint32
420,1351728,fVar6,"271,281",float,register,xmm1_da,4640,BUILTIN,float


In [5]:
raw_strip[raw_strip.FunctionStart==addr]

,FunctionStart,Name,Signature,Type,LocType,LocRegName,LocOffset,TypeCategory,TypeSeq
391,1351728,lVar1,"400,408,408,408",int64,register,rcx,8,BUILTIN,int64
392,1351728,puVar2,"405,408,408,408",uint64*,register,rdi,56,PTR,"PTR,uint64"
393,1351728,in_FS_OFFSET,"15,1739",int64,register,fs_offset,272,BUILTIN,int64
394,1351728,bVar3,"4,408",uchar,register,df,522,BUILTIN,uchar
395,1351728,fVar4,"271,281",float,register,xmm1_da,4640,BUILTIN,float
396,1351728,uVar5,"0,144",uint32,register,xmm1_db,4644,BUILTIN,uint32
397,1351728,uVar6,"0,144,191,281",uint32,register,xmm1_db,4644,BUILTIN,uint32
398,1351728,local_378,"505,548,865,908,1225,1268",uint64,stack,NaN,-888,BUILTIN,uint64
399,1351728,local_370,"555,915,1275",uint64,stack,NaN,-880,BUILTIN,uint64
400,1351728,local_368,"576,936,1296",uint64,stack,NaN,-872,BUILTIN,uint64


In [6]:
raw_dwarf[raw_dwarf.FunctionStart==addr]

,Name,Type,LocType,LocRegName,LocOffset,FunctionStart,FunctionName,TypeCategory,TypeSeq
104,width,int32,UndefinedLoc,NaN,NaN,1351728,init_game,BUILTIN,int32
105,height,int32,UndefinedLoc,NaN,NaN,1351728,init_game,BUILTIN,int32
106,player_pos,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"
107,player_halfsize,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"
108,player_col,c_aabb,UndefinedLoc,NaN,NaN,1351728,init_game,STRUCT,STRUCT
109,sword_pos,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"
110,sword_halfsize,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"
111,sword_col,c_aabb,UndefinedLoc,NaN,NaN,1351728,init_game,STRUCT,STRUCT
112,zero,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"
113,halfsize,float[2],UndefinedLoc,NaN,NaN,1351728,init_game,ARR,"ARR,float"


# Export DataTypeArchive
Ok, try this REALLY QUICK!
1. **I need this to be able to try out/investigate the member offset dataset!!**
2. If I can hand Dylan the `sword_swoosh` local from `init_game` that is a great test case

In [7]:
import pyhidra
pyhidra.start()

In [8]:
!ls ~/ghidra_projects

aarch64test.gpr  astera3.gpr  astera.gpr   astera.lock~
aarch64test.rep  astera3.rep  astera.lock  astera.rep


In [9]:
import typing
if typing.TYPE_CHECKING:
    import ghidra
    from ghidra.ghidra_builtins import *

from ghidralib.datatypes import to_varlib_dtype

# to_varlib_dtype()
import ghidra
from ghidra.base.project import GhidraProject

gproj = GhidraProject.openProject(Path.home()/'ghidra_projects', 'astera', False)

In [10]:
list(list(gproj.getRootFolder().getFolders())[0].getFiles())
prog = gproj.openProgram('/run1.gcc-O0.astera', '0.fighter.debug', True)

In [11]:
dtmgr = prog.getDataTypeManager()
s = dtmgr.getAllStructures().next()

In [12]:
stype = to_varlib_dtype(s, s.getLength())
stype.sid

# len([to_varlib_dtype(s, s.getLength()) for s in dtmgr.getAllStructures()])
[to_varlib_dtype(s, s.getLength()).name for s in dtmgr.getAllStructures()][:10]

['Elf64_Ehdr',
 'Elf64_Phdr',
 'Elf64_Shdr',
 'Elf64_Dyn',
 'Elf64_Sym',
 'Elf64_Rela',
 'NoteAbiTag',
 'NoteGnuProperty_4',
 'NoteGnuPropertyElement_4',
 '_IO_FILE']

In [13]:
from varlib.datatype import StructField

comp = list(s.getComponents())[0]
comp.getDataType()
comp.getOffset()
comp.getFieldName()
comp.getComment()
comp.getDefaultFieldName()
comp.getKey()
comp.getLength()
comp.getDataType().getLength()
comp.getOrdinal()
comp.getParent()
comp.getEndOffset()

f = StructField(to_varlib_dtype(comp.getDataType(), comp.getLength()), comp.getFieldName())
f.name
print(f)
print(f.name)
type(comp)

uchar e_ident_magic_num
e_ident_magic_num


<java class 'ghidra.program.database.data.DataTypeComponentDB'>

# Export Ghidra Data Type Archive
The code below needs to be moved to `ghidralib`

In [14]:
from ghidralib.export_types import export_ghidra_types_to_sdb

sdb = export_ghidra_types_to_sdb(dtmgr)
sdb.to_json('astera.debug.sdb')

100%|██████████| 406/406 [00:00<00:00, 1931.71it/s]


In [15]:
k = list(sdb.structs_by_id.keys())[35]
print(sdb.structs_by_id[k])

struct r_anim_viewer {
	0x0: r_anim* anim
	0x8: double time
	0x10: double rate
	0x18: uint32 curr
	0x1c: uint32 count
	0x20: uchar state
	0x21: uchar pstate
	0x22: char loop
}


In [16]:
!ls -Ahl

total 2.6M
-rw-rw-r-- 1 cls0027 cls0027 1.2M Feb  3 11:59 astera.debug.sdb
-rw-rw-r-- 1 cls0027 cls0027 1.1M Feb  1 19:00 astera.sdb
-rw-rw-r-- 1 cls0027 cls0027 3.9K Feb  1 10:47 ast_updates.ipynb
-rw-rw-r-- 1 cls0027 cls0027 281K Feb  3 11:35 dylan_sandbox.ipynb
drwxrwxr-x 8 cls0027 cls0027 4.0K Feb  2 20:42 .git
-rw-rw-r-- 1 cls0027 cls0027   34 Feb  1 19:11 .gitignore
-rw-rw-r-- 1 cls0027 cls0027   90 May 15  2023 pyproject.toml
-rw-rw-r-- 1 cls0027 cls0027   72 Aug 21 19:57 README.md
-rw-rw-r-- 1 cls0027 cls0027  524 Jan 30 20:39 setup.cfg
drwxrwxr-x 7 cls0027 cls0027 4.0K Jan 30 09:06 src


In [17]:
sdb2 = sdb.from_json('astera.debug.sdb')

In [18]:
sid = sdb2.sids_by_name['r_anim'][0]
sdb2.structs_by_id[sid]

struct r_anim {
	0x0: uint32 id
	0x8: uint32* frames
	0x10: double* lengths
	0x18: uint32 count
	0x20: double rate
	0x28: r_sheet* sheet
	0x30: char loop
}

In [19]:
sdb2.structs_by_id.keys()

dict_keys([72057594037927936, 72057594037927937, 72057594037927938, 72057594037927939, 72057594037927940, 72057594037927941, 72057594037927942, 72057594037927943, 72057594037927944, 72057594037927945, 72057594037927946, 72057594037927947, 72057594037927948, 72057594037927949, 72057594037927950, 72057594037927951, 72057594037927952, 72057594037927953, 72057594037927954, 72057594037927955, 72057594037927956, 72057594037927957, 72057594037927958, 72057594037927959, 72057594037927960, 72057594037927961, 72057594037927962, 72057594037927963, 72057594037927964, 72057594037927965, 72057594037927966, 72057594037927967, 72057594037927968, 72057594037927969, 72057594037927970, 72057594037927971, 72057594037927973, 72057594037927974, 72057594037927975, 72057594037927976, 72057594037927977, 72057594037927978, 72057594037927980, 72057594037927981, 72057594037927982, 72057594037927983, 72057594037927984, 72057594037927985, 72057594037927986, 72057594037927987, 72057594037927988, 72057594037927989, 7

In [20]:
locals_df[locals_df.FunctionStart==addr]

,FunctionStart,Signature,Name_Strip,Type_Strip,LocType_Strip,LocRegName_Strip,LocOffset_Strip,TypeCategory_Strip,TypeSeq_Strip,Name_Debug,Type_Debug,LocType_Debug,LocRegName_Debug,LocOffset_Debug,TypeCategory_Debug,TypeSeq_Debug,HasDWARF,TypeJson_Debug,BinaryId
373,1351728,"400,408,408,408",lVar1,int64,register,rcx,8,BUILTIN,int64,lVar3,int64,register,rcx,8.0,BUILTIN,int64,False,"{""kind"": ""BuiltinType"", ""name"": ""long"", ""is_fp...",0
374,1351728,"405,408,408,408",puVar2,uint64*,register,rdi,56,PTR,"PTR,uint64",ppVar4,player_t*,register,rdi,56.0,PTR,"PTR,STRUCT",False,"{""kind"": ""PointerType"", ""size"": 8, ""inner"": [{...",0
375,1351728,"4,408",bVar3,uchar,register,df,522,BUILTIN,uchar,bVar5,uchar,register,df,522.0,BUILTIN,uchar,False,"{""kind"": ""BuiltinType"", ""name"": ""byte"", ""is_fp...",0
376,1351728,"271,281",fVar4,float,register,xmm1_da,4640,BUILTIN,float,fVar6,float,register,xmm1_da,4640.0,BUILTIN,float,False,"{""kind"": ""BuiltinType"", ""name"": ""float"", ""is_f...",0
377,1351728,"0,144",uVar5,uint32,register,xmm1_db,4644,BUILTIN,uint32,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0
378,1351728,"0,144,191,281",uVar6,uint32,register,xmm1_db,4644,BUILTIN,uint32,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0
379,1351728,"505,548,865,908,1225,1268",local_378,uint64,stack,NaN,-888,BUILTIN,uint64,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0
380,1351728,"555,915,1275",local_370,uint64,stack,NaN,-880,BUILTIN,uint64,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0
381,1351728,"576,936,1296",local_368,uint64,stack,NaN,-872,BUILTIN,uint64,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0
382,1351728,"583,943,1303",local_360,uint64,stack,NaN,-864,BUILTIN,uint64,NaN,NaN,NaN,NaN,NaN,COMP,COMP,False,NaN,0


In [21]:
funcs_df[funcs_df.FunctionStart==addr]

,FunctionStart,FunctionName_Debug,AstJson_Debug,FunctionName_Strip,AstJson_Strip,FunctionName_DWARF,BinaryId
36,1351728,init_game,/home/cls0027/exp_builds/astera.exp/rundata/ru...,FUN_0014a030,/home/cls0027/exp_builds/astera.exp/rundata/ru...,init_game,0


In [39]:
# ghidra imports
from ghidra.program.model.listing import Program

# my stuff (astlib)
from varlib import datatype, StructDatabase

###################################################################
# STUB CLASS/FUNCTIONS (aka what I need you to implement)
# -----------------------------------------------------------------
# This is a specific example of the general idea I'm looking for...lol
#
# What I mean is, we can totally tweak the API - this is just my best
# guess at a concrete example of what I need
###################################################################

class GhidraRetyper:
    def __init__(self, program:Program, reference_db:StructDatabase) -> None:
        # NOTE: I think this is all the Ghidra context you'll need
        # to access database things but add more if you need to
        self.program = program

        # right now the db I'm handing you has everything, but
        # the only guarantee is that it has definitions for all of
        # the structure/union types I actually ask you to apply
        # (and their dependencies) so it may be a subset of
        # Ghidra's full data type archive in general
        self.reference_db = reference_db

    def define_all_reference_types(self, overwrite_existing:bool=False):

        # this is where you may need to perform a topological sort
        # or w/e to handle dependencies (structs within structs)
        # Jacob did this recently...may be good to ask him about it
        print(f'DYLAN TODO: make sure this works for nested structure/union types...')

        # Caleb's naive implementation:
        for sid, stype in self.reference_db.structs_by_id:
            self.define_struct_type(stype, overwrite_existing)

        for uid, utype in self.reference_db.unions_by_id:
            self.define_union_type(utype, overwrite_existing)

    def define_struct_type(self, stype:datatype.StructType, overwrite_existing:bool=False):
        # define the structure using its given name and layout
        # --> IGNORE THE SID (stype.sid). Ghidra will assign its own ID and I will
        #     probably re-export the data type archive and remap the ids outside of this
        #     Regardless, I won't assume the sids are still valid once this completes.
        #     I will only assume the structure with the given name is defined in the database

        # also...if Ghidra wants you to pick a path for the data types (that show up in
        # the data type manager tree on the left), we can just use some hardcoded path
        # for all type we define for now (e.g. "/GhidraRetyper")

        # if overwrite_existing then blow away an existing struct with the same name
        print(f'DYLAN TODO: define structure type {stype.name} in Ghidra')

    def define_union_type(self, utype:datatype.UnionType, overwrite_existing:bool=False):
        # same as above, but unions...
        print(f'DYLAN TODO: define union type {utype.name} in Ghidra')

    def set_localvar_type(self, func_addr:int, local_name:str, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler local {local_name} in {func_addr:#x} to type {dtype}')

    def set_globalvar_type(self, global_name:int, global_type:datatype.DataType):
        # do this last: I don't have data for globals right now and we may not need them
        # ...but, while you're doing the others if this is straightforward you can add
        # support for globals too
        pass

    def set_param_type(self, func_addr:int, param_name:str, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler parameter {param_name} in {func_addr:#x} to type {dtype}')

    def set_return_type(self, func_addr:int, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler return type for function {func_addr:#x} to type {dtype}')

###################################################################
# DRIVER CODE
# -----------------------------------------------------------------
# My test cases and a good example of how I will use your class
###################################################################


# --------------------------------
# CALEB TODO:

# save/restore data types via TypeJson_Debug <<<

# --------------------------------
# STRUCT - include sid
# PTR - pointer, keep going...
# uint64, uchar, etc. - convert from standard name...

from varlib.datatype import datatype_from_dict
import json

retyper = GhidraRetyper(prog, sdb2)
func_locals = locals_df[locals_df.FunctionStart==addr]

for i in range(len(locals_df)):
    x = locals_df.iloc[i]
    if x.TypeCategory_Debug == 'COMP':
        continue
    # retyper.set_localvar_type(addr, x.Name_Strip, x.Type_Debug)
    try:
        print(i, x.TypeSeq_Debug, f'{datatype_from_dict(json.loads(x.TypeJson_Debug), sdb2)}')
    except KeyError:
        print(f'Failed on {i}, {x.TypeJson_Debug}')

0 ARR,uchar uchar[8]
1 int32 int32
2 int32 int32
3 int32 int32
7 int32 int32
8 int32 int32
9 float float
10 int32 int32
11 int32 int32
12 float float
13 int32 int32
14 int32 int32
16 PTR,STRUCT asset_t*
17 PTR,STRUCT asset_t*
18 PTR,STRUCT asset_t*
25 float float
26 int32 int32
27 int32 int32
28 double double
29 double double
31 float float
32 double double
33 uchar uchar
34 PTR,STRUCT r_particles*
35 int32 int32
37 float float
47 PTR,STRUCT r_sheet*
48 uint64 uint64
49 uint64 uint64
50 uint64 uint64
51 uint64 uint64
52 uint64 uint64
53 uint64 uint64
54 uint64 uint64
56 uint64 uint64
57 uint64 uint64
58 uint64 uint64
59 uint64 uint64
60 uint64 uint64
61 uint64 uint64
62 uint64 uint64
63 uint64 uint64
64 int32 int32
67 int32 int32
68 float float
70 int32 int32
72 float float
74 PTR,STRUCT r_anim*
84 int32 int32
86 float float
88 ARR,float float[2]
94 int64 int64
95 PTR,uint64 uint64*
96 uchar uchar
97 float float
98 float float
99 float float
100 float float
104 ARR,float float[2]
108 u

In [23]:
x = locals_df.iloc[6584]

# dtmgr.findDataTypeForID(ghidra.util.UniversalID(360287970189640893))
# type(dtmgr.findDataType('_XRRScreenResources'))
funcs_df[funcs_df.FunctionStart==1952083]
x

FunctionStart                                                   1952083
Signature                                                   114,169,267
Name_Strip                                                        uVar1
Type_Strip                                                       uint64
LocType_Strip                                                  register
LocRegName_Strip                                                    rax
LocOffset_Strip                                                       0
TypeCategory_Strip                                              BUILTIN
TypeSeq_Strip                                                    uint64
Name_Debug                                                       pXVar1
Type_Debug                                          XRRScreenResources*
LocType_Debug                                                  register
LocRegName_Debug                                                    rax
LocOffset_Debug                                                 

In [24]:
td = list(dtmgr.getAllDataTypes())[0]

In [37]:
# OK, currently the AST will contain typedef type ids and names for typedef'd structs
# and unions, but defines them as the canonical struct/union definition
# --> if we simply double-map the ids/names, everything should work fine...

isinstance(td, ghidra.program.model.data.TypeDef)

typedef_types = [x for x in dtmgr.getAllDataTypes() if isinstance(x, ghidra.program.model.data.TypeDef)]
print(len(typedef_types))
print(len([td for td in typedef_types if isinstance(td.getBaseDataType(), ghidra.program.model.data.Structure)]))
utype = [td for td in typedef_types if isinstance(td.getBaseDataType(), ghidra.program.model.data.Union)][0].getBaseDataType()
td = [td for td in typedef_types if isinstance(td.getBaseDataType(), ghidra.program.model.data.Union)][0]
utype

1231
50


/DWARF/Xlib.h/_XEvent
pack()
Union _XEvent {
   0   int   4   type   ""
   0   XAnyEvent   40   xany   ""
   0   XKeyEvent   96   xkey   ""
   0   XButtonEvent   96   xbutton   ""
   0   XMotionEvent   96   xmotion   ""
   0   XCrossingEvent   104   xcrossing   ""
   0   XFocusChangeEvent   48   xfocus   ""
   0   XExposeEvent   64   xexpose   ""
   0   XGraphicsExposeEvent   72   xgraphicsexpose   ""
   0   XNoExposeEvent   48   xnoexpose   ""
   0   XVisibilityEvent   48   xvisibility   ""
   0   XCreateWindowEvent   72   xcreatewindow   ""
   0   XDestroyWindowEvent   48   xdestroywindow   ""
   0   XUnmapEvent   56   xunmap   ""
   0   XMapEvent   56   xmap   ""
   0   XMapRequestEvent   48   xmaprequest   ""
   0   XReparentEvent   72   xreparent   ""
   0   XConfigureEvent   88   xconfigure   ""
   0   XGravityEvent   56   xgravity   ""
   0   XResizeRequestEvent   48   xresizerequest   ""
   0   XConfigureRequestEvent   96   xconfigurerequest   ""
   0   XCirculateEvent   56   x

In [38]:
sdb2.unions_by_id[td.key]

union _XEvent {
	[int32 type, XAnyEvent xany, XKeyEvent xkey, XButtonEvent xbutton, XMotionEvent xmotion, XCrossingEvent xcrossing, XFocusChangeEvent xfocus, XExposeEvent xexpose, XGraphicsExposeEvent xgraphicsexpose, XNoExposeEvent xnoexpose, XVisibilityEvent xvisibility, XCreateWindowEvent xcreatewindow, XDestroyWindowEvent xdestroywindow, XUnmapEvent xunmap, XMapEvent xmap, XMapRequestEvent xmaprequest, XReparentEvent xreparent, XConfigureEvent xconfigure, XGravityEvent xgravity, XResizeRequestEvent xresizerequest, XConfigureRequestEvent xconfigurerequest, XCirculateEvent xcirculate, XCirculateRequestEvent xcirculaterequest, XPropertyEvent xproperty, XSelectionClearEvent xselectionclear, XSelectionRequestEvent xselectionrequest, XSelectionEvent xselection, XColormapEvent xcolormap, XClientMessageEvent xclient, XMappingEvent xmapping, XErrorEvent xerror, XKeymapEvent xkeymap, XGenericEvent xgeneric, XGenericEventCookie xcookie, int64[24] pad]
}

In [26]:
sdb2.unions_by_id[utype.key]

union _XEvent {
	[int32 type, XAnyEvent xany, XKeyEvent xkey, XButtonEvent xbutton, XMotionEvent xmotion, XCrossingEvent xcrossing, XFocusChangeEvent xfocus, XExposeEvent xexpose, XGraphicsExposeEvent xgraphicsexpose, XNoExposeEvent xnoexpose, XVisibilityEvent xvisibility, XCreateWindowEvent xcreatewindow, XDestroyWindowEvent xdestroywindow, XUnmapEvent xunmap, XMapEvent xmap, XMapRequestEvent xmaprequest, XReparentEvent xreparent, XConfigureEvent xconfigure, XGravityEvent xgravity, XResizeRequestEvent xresizerequest, XConfigureRequestEvent xconfigurerequest, XCirculateEvent xcirculate, XCirculateRequestEvent xcirculaterequest, XPropertyEvent xproperty, XSelectionClearEvent xselectionclear, XSelectionRequestEvent xselectionrequest, XSelectionEvent xselection, XColormapEvent xcolormap, XClientMessageEvent xclient, XMappingEvent xmapping, XErrorEvent xerror, XKeymapEvent xkeymap, XGenericEvent xgeneric, XGenericEventCookie xcookie, int64[24] pad]
}

In [27]:
ss = sdb2.sids_by_name['_XRRScreenResources'][0]
print(sdb2.structs_by_id[ss].ghidra_uid)
print(360287970189640721)


3579440783260282748
360287970189640721


In [28]:
from ghidralib.decompiler import get_decompiler_interface

ifc = get_decompiler_interface(prog)

In [29]:
ff = [x for x  in prog.functionManager.getFunctions(True) if x.getEntryPoint().offset==addr][0]
print(ff.name)
res = ifc.decompileFunction(ff, 240, None)

init_game


In [30]:
# this works!
xx = res.getHighFunction().getDataTypeManager().findBaseType('player_t', 72057594037927994)
(xx.getUniversalID(), xx.getKey())


(3579440781830025081, 72057594037927994)

In [31]:
xx.getUniversalID().getValue()
hex(72057594037927949)

'0x10000000000000d'

In [32]:
# can I just create a 1-time map of all struct types (using an arbitrary decompiled func?)
k = list(sdb2.structs_by_id.keys())[1]

sdef = sdb2.structs_by_id[k]
k, sdef.name

res.getHighFunction().getDataTypeManager().findBaseType(sdef.name, k)

/ELF/Elf64_Phdr
pack(disabled)
Structure Elf64_Phdr {
   0   Elf_ProgramHeaderType   4   p_type   ""
   4   dword   4   p_flags   ""
   8   qword   8   p_offset   ""
   16   qword   8   p_vaddr   ""
   24   qword   8   p_paddr   ""
   32   qword   8   p_filesz   ""
   40   qword   8   p_memsz   ""
   48   qword   8   p_align   ""
}
Size = 56   Actual Alignment = 1

In [33]:
import fixedint

dsid = 72057594037927994
sname = 'player_t'
# gsid = 3579440781830025081

# can I generate dsid from Ghidra name?

res = fixedint.UInt64(123)

# res = 123
for c in sname:
    res = fixedint.UInt64((res<<8) | (res>>56))
    res += fixedint.UInt64(ord(c))
    if (res&1) == 0:
        res ^= fixedint.UInt64(0xfeabfeab)
tmp = fixedint.UInt64(1)
tmp <<= 63
res |= tmp

print(res)
res == dsid
hex(dsid)
dsid==xx.getKey()

11945691551399600393


True

In [34]:
sdef.layout.fields_by_offset[0x30].size+0x30

sizehash = fixedint.UInt64(56)
sizehash *= fixedint.UInt64(0x98251033aecbabaf)
res ^= sizehash
res
# res ^ sizehash


# sizehash = fixedint.UInt64(56)
# sizehash ^ fixedint.UInt64(0x98251033aecbabaf)

UInt64(17139596697048149825)

In [35]:
tmp = fixedint.UInt64(1)
tmp <<= 63
hex(tmp)
type(tmp)
type(tmp<<8)

fixedint.aliases.UInt64

In [36]:
#!pip install fixedint